In [1]:
import pandas as pd

# 1. Cargar datos del DENUE
# Descarga los CSVs del DENUE (Sector 46: Comercio al por menor y Sector 72: Alimentos/Bebidas)
# o ajusta la ruta a tu archivo DENUE unificado.
ruta_denue = 'data/raw/denue_comercio.csv'  
df_denue = pd.read_csv(ruta_denue, encoding='latin1', low_memory=False)

# 2. Filtrar Zonas de Estudio (Jalisco=14, Nuevo León=19, Querétaro=22)
# 'cve_ent' o los primeros 2 dígitos de 'clee' / 'ubica_geo'
df_denue['entidad_id'] = df_denue['cve_ent'].astype(str).str.zfill(2)
estados_target = ['14', '19', '22']
df_denue = df_denue[df_denue['entidad_id'].isin(estados_target)].copy()

# Crear código de municipio estandarizado (5 dígitos: Entidad + Municipio)
df_denue['ubica_geo'] = (
    df_denue['cve_ent'].astype(str).str.zfill(2) + 
    df_denue['cve_mun'].astype(str).str.zfill(3)
)

# 3. Identificar Competencia Directa e Indirecta (Código SCIAN)
# 462111: Comercio al por menor en supermercados
# 462112: Comercio al por menor en mini supers / tiendas de conveniencia
# Filtramos palabras clave para detectar Sam's Club, City Club y Costco existente

nom_comercial = df_denue['nom_estab'].str.upper().fillna('')

# Competidores Directos (Formatos Club)
is_sams = nom_comercial.str.contains("SAM'S|SAMS")
is_cityclub = nom_comercial.str.contains("CITY CLUB")
is_costco = nom_comercial.str.contains("COSTCO")

df_denue['es_competencia_club'] = np.where(is_sams | is_cityclub, 1, 0)
df_denue['costco_existente'] = np.where(is_costco, 1, 0)

# Masa B2B (Restaurantes, Hoteles, Minisúpers -> Compradores de Membresía Business)
# Códigos SCIAN 722 (Servicios de preparación de alimentos y bebidas) y 462112
codigo_scian = df_denue['codigo_scian'].astype(str)
is_b2b = codigo_scian.str.startswith(('722', '462112', '461110'))
df_denue['es_cliente_b2b'] = np.where(is_b2b, 1, 0)

# 4. Agrupar Variables de Oferta por Municipio
oferta_municipal = df_denue.groupby('ubica_geo').agg(
    competidores_club=('es_competencia_club', 'sum'),
    costcos_actuales=('costco_existente', 'sum'),
    densidad_b2b=('es_cliente_b2b', 'sum'),
    total_unidades_economicas=('id', 'count')
).reset_index()

print("Mapeo de oferta completado.")
display(oferta_municipal.sort_values(by='competidores_club', ascending=False).head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/denue_comercio.csv'